# Phase 1 Robustness Checks

Supplementary analyses validating the main DiD and Causal Forest results.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/cleaned/final_analysis_data.csv')

print(f"Shape: {df.shape}")
print(f"Treated obs (post_carbon_tax): {df['post_carbon_tax'].sum()}")
print(f"Treatment cohorts: {sorted(df.loc[df['treatment_year'].notna(), 'treatment_year'].unique().astype(int))}")
print(f"Never-treated countries: {df[df['treatment_year'].isna()]['country'].nunique()}")

## 1. Sun-Abraham Staggered-Robust DiD

Standard TWFE with 13 treatment cohorts is vulnerable to negative weight bias when treatment effects are heterogeneous (Goodman-Bacon 2021). Sun & Abraham (2021) estimates cohort-specific ATTs and aggregates them correctly.

In [ ]:
import pyfixest as pf

# Prepare data for pyfixest sunab()
df_staggered = df.copy()

# Never-treated: cohort must be a value that never appears as a year
# pyfixest sunab convention: use a large number for never-treated
df_staggered['cohort'] = df_staggered['treatment_year'].fillna(10000).astype(int)

# Encode country as integer (required by pyfixest for FE)
df_staggered['country_id'] = df_staggered['country'].astype('category').cat.codes

# Use co2_per_capita_future_trend as primary outcome (matches CLAUDE.md)
# Fall back to co2_per_capita_3yr_change if future_trend not available
outcome = 'co2_per_capita_future_trend'
if outcome not in df_staggered.columns or df_staggered[outcome].isna().all():
    outcome = 'co2_per_capita_3yr_change'

# Drop rows with missing outcome
df_staggered = df_staggered[df_staggered[outcome].notna()].copy()

print(f"Staggered sample: {df_staggered.shape}")
print(f"Cohorts: {sorted(df_staggered['cohort'].unique())}")
print(f"Outcome variable: {outcome}")

In [ ]:
# Sun-Abraham (2021) — staggered-robust, no negative weights
fit_sa = pf.feols(
    f"{outcome} ~ sunab(cohort, year) | country_id + year",
    data=df_staggered,
    vcov={"CRV1": "country_id"}
)

print(fit_sa.summary())

In [ ]:
import os
os.makedirs('../outputs', exist_ok=True)

# Event study plot from Sun-Abraham
fig = fit_sa.iplot(
    coord_flip=False,
    figsize=(12, 6),
    title="Sun-Abraham Event Study (Staggered-Robust)",
).get_figure()
fig.savefig('../outputs/sun_abraham_event_study.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to outputs/sun_abraham_event_study.png")

In [ ]:
# Aggregate Sun-Abraham to overall ATT
try:
    sa_agg = fit_sa.aggregate(agg="ATT")
    print("=== Aggregated ATT (Sun-Abraham) ===")
    print(sa_agg.summary())
except Exception as e:
    print(f"Aggregation method: {e}")
    # Manual aggregation fallback — average post-treatment coefficients
    sa_coefs = fit_sa.coef()
    post_coefs = {k: v for k, v in sa_coefs.items() if '::' in k}
    if post_coefs:
        mean_att = np.mean(list(post_coefs.values()))
        print(f"\nManual average of post-treatment cohort-time ATTs: {mean_att:.4f}")

# Compare with standard TWFE
fit_twfe = pf.feols(
    f"{outcome} ~ post_carbon_tax + log_gdp + log_population + "
    f"trade_openness + natural_resource_rents_per_gdp + fossil_pct_filled | country_id + year",
    data=df_staggered.dropna(subset=['log_gdp', 'log_population', 'trade_openness',
                                      'natural_resource_rents_per_gdp', 'fossil_pct_filled']),
    vcov={"CRV1": "country_id"}
)

print("\n=== TWFE Estimate (for comparison) ===")
print(f"  Coefficient: {fit_twfe.coef()['post_carbon_tax']:.4f}")
print(f"  SE:          {fit_twfe.se()['post_carbon_tax']:.4f}")
print(f"  p-value:     {fit_twfe.pvalue()['post_carbon_tax']:.4f}")

## Staggered DiD Robustness Check

| Estimator | ATT Estimate | SE | Notes |
|-----------|-------------|-----|-------|
| TWFE (standard) | fill in | fill in | Original spec, susceptible to negative weights |
| Sun-Abraham (2021) | fill in | fill in | Staggered-robust, no negative weights |

**Conclusion:** fill in after running

## 2. ETS Control Group Sensitivity

EU ETS countries (carbon pricing via cap-and-trade but no carbon tax) are in the control group. This contaminates "untreated" status and may bias the treatment effect downward. Below we test three control group definitions.

In [ ]:
# Identify ETS countries in control group
print("ETS column check:")
print(df['has_ets'].value_counts())
print(f"\nETS countries in control group (has_ets=1, post_carbon_tax=0):")
ets_control = df[(df['has_ets'] == 1) & (df['post_carbon_tax'] == 0)]['country'].unique()
print(sorted(ets_control))
print(f"\nCount: {len(ets_control)} countries")

In [ ]:
import statsmodels.formula.api as smf

DID_FORMULA = """
co2_per_capita_future_trend ~ post_carbon_tax
    + C(year) + C(country)
    + log_gdp + log_population + trade_openness
    + natural_resource_rents_per_gdp + fossil_pct_filled
"""

def run_did(data, label):
    """Run DiD with clustered SEs and return results dict."""
    model = smf.ols(DID_FORMULA, data=data.dropna(
        subset=['co2_per_capita_future_trend', 'post_carbon_tax', 'log_gdp',
                'log_population', 'trade_openness', 'natural_resource_rents_per_gdp',
                'fossil_pct_filled']
    )).fit(
        cov_type='cluster', cov_kwds={'groups': data.dropna(
            subset=['co2_per_capita_future_trend', 'post_carbon_tax', 'log_gdp',
                    'log_population', 'trade_openness', 'natural_resource_rents_per_gdp',
                    'fossil_pct_filled']
        )['country']}
    )
    coef = model.params['post_carbon_tax']
    se = model.bse['post_carbon_tax']
    pval = model.pvalues['post_carbon_tax']
    n_countries = data['country'].nunique()
    n_obs = int(model.nobs)
    print(f"  {label:<45} coef={coef:7.4f}  SE={se:.4f}  p={pval:.3f}  "
          f"N={n_obs}  countries={n_countries}")
    return {'label': label, 'coef': coef, 'se': se, 'pval': pval, 'n': n_obs}

# Spec A: original (ETS countries in control)
df_spec_a = df.copy()

# Spec B: exclude ETS-only countries from control group
# Keep: treated (carbon tax) + pure never-treated (no tax, no ETS)
df_spec_b = df[~((df['has_ets'] == 1) & (df['post_carbon_tax'] == 0))].copy()

# Spec C: only never-treated controls (no carbon tax AND no ETS ever)
never_any_pricing = df.groupby('country').apply(
    lambda x: (x['has_ets'].max() == 0) and (x['post_carbon_tax'].max() == 0)
)
never_any_countries = never_any_pricing[never_any_pricing].index.tolist()
treated_countries = df[df['post_carbon_tax'] == 1]['country'].unique().tolist()
df_spec_c = df[df['country'].isin(never_any_countries + treated_countries)].copy()

print("=== ETS Sensitivity Analysis ===\n")
results_a = run_did(df_spec_a, "A: Original (ETS in control)")
results_b = run_did(df_spec_b, "B: Exclude ETS-only countries")
results_c = run_did(df_spec_c, "C: Never-any-pricing controls only")

In [ ]:
results = [results_a, results_b, results_c]
labels = [r['label'] for r in results]
coefs = [r['coef'] for r in results]
ses = [r['se'] for r in results]

fig, ax = plt.subplots(figsize=(9, 5))
y_pos = np.arange(len(results))

ax.errorbar(coefs, y_pos, xerr=[1.96 * s for s in ses],
            fmt='o', color='steelblue', capsize=5, markersize=8, linewidth=2)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_yticks(y_pos)
ax.set_yticklabels(labels, fontsize=10)
ax.set_xlabel('DiD Coefficient (Effect on CO₂/Capita Trend)', fontsize=11)
ax.set_title('ETS Control Group Sensitivity\n(All specs cluster SEs at country level)', fontsize=12)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/ets_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to outputs/ets_sensitivity.png")

## ETS Control Group Sensitivity

| Spec | Description | Coef | SE | p-value |
|------|-------------|------|----|---------|
| A | Original (ETS in control) | fill in | fill in | fill in |
| B | Exclude ETS-only countries | fill in | fill in | fill in |
| C | Never-any-pricing controls only | fill in | fill in | fill in |

**Finding:** fill in after running

**Preferred specification for Paper 1:** fill in